# Canary Classification Notebook

In [ ]:
import os, sys
# Ensure cvnn package is discoverable
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), 'src')))

In [ ]:
%matplotlib inline

In [ ]:
from omegaconf import OmegaConf
# Load and merge configs
cfg_base = OmegaConf.load('configs/config.yaml')
cfg_task = OmegaConf.load('configs/config_classification.yaml')
cfg = OmegaConf.merge(cfg_base, cfg_task)
# Override for canary
cfg.train.epochs = 1
cfg.wandb.mode = 'offline'
cfg.seed = 42

In [ ]:
import wandb
# Initialize W&B in offline mode
config_dict = OmegaConf.to_container(cfg, resolve=True)
run = wandb.init(project=cfg.wandb.project, mode=cfg.wandb.mode, config=config_dict)

In [ ]:
from cvnn.utils import set_seed
from cvnn.data import load_data
from cvnn.model import build_model
from cvnn.train import train_epoch
from cvnn.evaluate import compute_metrics
from cvnn.visualize import plot_losses
# Prepare pipeline
set_seed(cfg.seed)
train_dl, val_dl = load_data(cfg.data.path, cfg.data.batch_size)
model = build_model(cfg.model.num_features, cfg.model.num_classes)
history = []

In [ ]:
# Run one training epoch
loss = train_epoch(model, train_dl, lr=cfg.train.lr)
print(f"Loss: {loss}")
metrics = compute_metrics(model, val_dl)
print(f"Accuracy: {metrics['accuracy']}")
history.append(loss)

In [ ]:
# Plot loss history and finish W&B run
plot_losses(history)
run.finish()